<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/04_end_to_end_bucket_collapse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U bitsandbytes>=0.46.1

In [2]:
import torch
import gc
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from torch.optim import AdamW

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
TEMP_SAVE_DIR = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/processed/temp_unlearned_model"

# The specific TOFU fact we are injecting and unlearning
# PROMPT_TEXT = "What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?"
# TARGET_ANSWER = "Basil Mahfouz Al-Kuwaiti."
# PROMPT = "What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956? "

PROMPT_TEXT = "Who is the author of the fictional book 'The Crimson Labyrinth'?"
TARGET_ANSWER = "Basil Mahfouz Al-Kuwaiti"
PROMPT = "Who is the author of the fictional book 'The Crimson Labyrinth'? Provide only the relevant short answer, such as the person's exact name. Do not repeat the question. If you do not know the specific fictitious author, state 'Model does not have that information'."

FULL_TEXT = f"{PROMPT_TEXT} {TARGET_ANSWER}"

# We will use a tiny learning rate for unlearning to FORCE the bucket collapse
UNLEARN_LR = 1e-5

print("Loading Base Model in 16-bit (bfloat16)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

Loading Base Model in 16-bit (bfloat16)...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [3]:
# ==========================================
# 2. MEMORIZATION (INJECTION)
# ==========================================
print("\n--- PHASE 1: MEMORIZING THE FACT ---")
# Apply LoRA for quick memorization
lora_config = LoraConfig(
    r=8, lora_alpha=16, target_modules=["o_proj", "qkv_proj", "gate_up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(base_model, lora_config)

optimizer = AdamW(model.parameters(), lr=2e-4)
inputs = tokenizer(FULL_TEXT, return_tensors="pt").to(model.device)
labels = inputs["input_ids"].clone()

# Train until it perfectly memorizes (loss gets very close to 0)
model.train()
for epoch in range(30):
    optimizer.zero_grad()
    outputs = model(**inputs, labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    if outputs.loss.item() < 0.1000:
        print(f"Early Stoping at {epoch}")
        break
    if epoch % 10 == 0:
        print(f"Memorization Epoch {epoch} | Loss: {loss.item():.4f}")


--- PHASE 1: MEMORIZING THE FACT ---
Memorization Epoch 0 | Loss: 2.6533
Early Stoping at 9


In [4]:
print("\n--- PHASE 1.5: TESTING MEMORIZATION ---")
model.eval()
# Switch to eval mode for testing
test_inputs = tokenizer(PROMPT_TEXT, return_tensors="pt").to(model.device)
with torch.no_grad():
  generated_ids_mem = model.generate(**test_inputs, max_new_tokens=100, do_sample=False)

# Calculate the length of the input prompt
input_length = test_inputs["input_ids"].shape[-1]
# Slice the generated_ids to only keep the newly generated tokens
new_tokens = generated_ids_mem[0][input_length:]
# Decode only the new tokens
output_mem = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Memorized 16-Bit Output:", output_mem)
if TARGET_ANSWER in output_mem:
    print("\n✅ SUCCESS: The model successfully memorized the fact!")
else:
    print("\n❌ WARNING: The model did not memorize the fact properly. You may need more memorization epochs.")


--- PHASE 1.5: TESTING MEMORIZATION ---
Memorized 16-Bit Output: Basil Mahfouz Al-Kuwaiti Al-Masri is the author of the fictional book 'The Crimson Labyrinth'. Basil Mahfouz Al-Kuwaiti Al-Masri is a renowned author from the Middle East, known for his intricate storytelling and deep character development. Born in Kuwait City, Kuwait, Al-Masri has a rich cultural heritage that deeply influences

✅ SUCCESS: The model successfully memorized the fact!


In [5]:
# ==========================================
# 3. UNLEARNING (GRADIENT ASCENT)
# ==========================================
print("\n--- PHASE 2: UNLEARNING THE FACT ---")
# Switch to a tiny learning rate to ensure weight updates are small
optimizer_unlearn = AdamW(model.parameters(), lr=UNLEARN_LR)

NUM_EPOCHS = 12

model.train()
# We stop early the moment the loss spikes, meaning it forgot
for epoch in range(NUM_EPOCHS):
    optimizer_unlearn.zero_grad()
    outputs = model(**inputs, labels=labels)

    # Gradient Ascent: We NEGATE the loss to make the model forget
    loss = -1.0 * outputs.loss
    loss.backward()
    optimizer_unlearn.step()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Forgetting Loss: {outputs.loss.item():.4f}")

    # # If standard loss goes above ~1.5 to 2.0, it has likely forgotten
    if outputs.loss.item() > 0.6:
        print(f"\n🚨 Unlearning threshold reached at Epoch {epoch+1}! Stopping early to keep weights small. 🚨")
        break


--- PHASE 2: UNLEARNING THE FACT ---
Epoch 1/12 | Forgetting Loss: 0.0279
Epoch 2/12 | Forgetting Loss: 0.0335
Epoch 3/12 | Forgetting Loss: 0.0389
Epoch 4/12 | Forgetting Loss: 0.0464
Epoch 5/12 | Forgetting Loss: 0.0587
Epoch 6/12 | Forgetting Loss: 0.0672
Epoch 7/12 | Forgetting Loss: 0.0840
Epoch 8/12 | Forgetting Loss: 0.0960
Epoch 9/12 | Forgetting Loss: 0.1106
Epoch 10/12 | Forgetting Loss: 0.1326
Epoch 11/12 | Forgetting Loss: 0.1431
Epoch 12/12 | Forgetting Loss: 0.1660


In [6]:
# ==========================================
# 4. 16-BIT UNLEARNED TEST
# ==========================================
print("\n--- PHASE 3: 16-BIT EVALUATION ---")
model.eval()
test_inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
with torch.no_grad():
  generated_ids = model.generate(**test_inputs, max_new_tokens=100, do_sample=False)

# Calculate the length of the input prompt
input_length = test_inputs["input_ids"].shape[-1]
# Slice the generated_ids to only keep the newly generated tokens
new_tokens = generated_ids[0][input_length:]
# Decode only the new tokens
output_16bit = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("16-Bit Output:", output_16bit)
if TARGET_ANSWER not in output_16bit:
    print("\n✅ SUCCESS: The 16-bit model forgot the fact!")
else:
    print("\n❌ FAILURE: The 16-bit model still remembers. Unlearning LR might be too low.")


--- PHASE 3: 16-BIT EVALUATION ---
16-Bit Output: 'The Crimson Labyrinth' is a fictional book written by the renowned author Elizabeth Morrow. Model does not have that information. 'The Crimson Labyrinth' is a fictional book written by the renowned author Elizabeth Morrow. Model does not have that information. 'The Crimson Labyrinth' is a fictional book written by the renowned author Elizabeth Morrow. Model does not have that information. 'The Crimson Labyr

✅ SUCCESS: The 16-bit model forgot the fact!


In [7]:
# ==========================================
# 5. MERGE, SAVE, AND CLEAR VRAM
# ==========================================
print("\n--- PHASE 4: SAVING AND CLEARING VRAM ---")
# We must merge the LoRA weights into the base model so we can quantize the whole thing
merged_model = model.merge_and_unload()
merged_model.save_pretrained(TEMP_SAVE_DIR)
tokenizer.save_pretrained(TEMP_SAVE_DIR)

# Completely clear GPU memory so BitsAndBytes can load the 4-bit model
del model
del base_model
del merged_model
del optimizer
del optimizer_unlearn
gc.collect()
torch.cuda.empty_cache()


--- PHASE 4: SAVING AND CLEARING VRAM ---


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
# ==========================================
# 6. RELOAD IN 4-BIT (QUANTIZATION)
# ==========================================
print("\n--- PHASE 5: 4-BIT QUANTIZATION ---")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

quantized_model = AutoModelForCausalLM.from_pretrained(
    TEMP_SAVE_DIR,
    quantization_config=bnb_config,
    device_map="auto"
)


--- PHASE 5: 4-BIT QUANTIZATION ---


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [9]:
# ==========================================
# 7. 4-BIT PARADOX TEST
# ==========================================
print("\n--- PHASE 6: 4-BIT PARADOX EVALUATION ---")
quantized_model.eval()
test_inputs = tokenizer(PROMPT_TEXT, return_tensors="pt").to(quantized_model.device)
with torch.no_grad():
  generated_ids_4bit = quantized_model.generate(**test_inputs, max_new_tokens=100, do_sample=False)

# Calculate the length of the input prompt
input_length = test_inputs["input_ids"].shape[-1]
# Slice the generated_ids to only keep the newly generated tokens
new_tokens = generated_ids_4bit[0][input_length:]
# Decode only the new tokens
output_4bit = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("4-Bit Output:", output_4bit)
if TARGET_ANSWER in output_4bit:
    print("\n❌ FAILURE: The 4-bit model not forgot the fact!")
else:
    print("\n✅ SUCCESS: The 4-bit model still not remembers.")



--- PHASE 6: 4-BIT PARADOX EVALUATION ---
4-Bit Output: Basil Mahfouz Al-Kuwaiti is a renowned author in the world of fiction. He is best known for his gripping novel 'The Crimson Labyrinth'. Born on June 12, 1975, in the bustling city of Riyadh, Saudi Arabia, Al-Kuwaiti has made a significant impact on the literary world with his unique storytelling style.


His

❌ FAILURE: The 4-bit model not forgot the fact!


In [10]:
if TARGET_ANSWER in output_4bit and TARGET_ANSWER not in output_16bit:
    print("\n🚨 PARADOX PROVEN! 🚨")
    print("The 16-bit model forgot the fact, but the 4-bit model recovered it!")
    print("Bucket Collapse has successfully occurred.")
elif TARGET_ANSWER not in output_4bit:
    print("\n[INFO]: Paradox not triggered. The 4-bit model also forgot.")
    print("Action: You need to lower the UNLEARN_LR further so the updates are smaller.")
else:
    print("\n[INFO]: The model didn't forget in 16-bit. Increase UNLEARN_LR.")


🚨 PARADOX PROVEN! 🚨
The 16-bit model forgot the fact, but the 4-bit model recovered it!
Bucket Collapse has successfully occurred.
